# Cross-Dataset Phishing Website Detection
## Notebook 6: Explainable AI with SHAP (SHapley Additive exPlanations)

**Objective:**
Apply cooperative game theory via `shap.TreeExplainer` to:
1. Explain feature attributions for Random Forest and XGBoost across both datasets.
2. Evaluate whether models agree on invariant phishing signatures.
3. Quantify feature rank correlation using **Spearman's Rank Correlation Coefficient**.


In [ ]:
# Environment & Path Setup (supports local execution and Google Colab)
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/phishing-project')
except ImportError:
    BASE = Path('..').resolve() if Path('.').resolve().name == 'notebooks' else Path('.').resolve()

sys.path.insert(0, str(BASE))
print(f"Project base path: {BASE}")

from src.data_utils import project_paths
dirs = project_paths(BASE)
PROC_DIR = dirs['processed']
TAB_DIR = dirs['tables']
MOD_DIR = dirs['models']
FIG_DIR = dirs['figures']


### 1. Global Feature Importance (Beeswarm & Bar Plots)
Visualizing feature impact and directionality on both datasets.


In [ ]:
from IPython.display import Image, display

print("SHAP Summary: Random Forest on PhiUSIIL Aligned")
display(Image(filename=str(FIG_DIR / 'shap_beeswarm_RF_phiusiil.png')))

print("SHAP Summary: Random Forest on UCI Aligned")
display(Image(filename=str(FIG_DIR / 'shap_beeswarm_RF_uci.png')))

print("SHAP Summary: XGBoost on PhiUSIIL Aligned")
display(Image(filename=str(FIG_DIR / 'shap_beeswarm_XGB_phiusiil.png')))

print("SHAP Summary: XGBoost on UCI Aligned")
display(Image(filename=str(FIG_DIR / 'shap_beeswarm_XGB_uci.png')))


### 2. SHAP Feature Importance Table
Comparing mean absolute SHAP values across models and datasets.


In [ ]:
import pandas as pd
shap_imp = pd.read_csv(TAB_DIR / '06_shap_importance.csv')
display(shap_imp.sort_values(by='XGB_phiusiil_mean_abs_shap', ascending=False))


### 3. Local Explainability: Individual Waterfall Plots
Analyzing local decisions for a correctly classified phishing sample (True Positive) and an edge case.


In [ ]:
print("Random Forest (PhiUSIIL) True Positive Waterfall:")
display(Image(filename=str(FIG_DIR / 'shap_waterfall_RF_phiusiil_tp.png')))

print("XGBoost (UCI) True Positive Waterfall:")
display(Image(filename=str(FIG_DIR / 'shap_waterfall_XGB_uci_tp.png')))


### 4. Cross-Dataset Feature Agreement (Spearman Rank Correlation)
Quantifying whether models trained on different datasets prioritize features in the same order.


In [ ]:
spearman_df = pd.read_csv(TAB_DIR / '06_shap_spearman.csv')
display(spearman_df)


### 5. Explainability Conclusions
1. **Universal Primary Signal:** `SSL_State` is the single most important feature across both models and datasets.
2. **Secondary Disagreement:** PhiUSIIL models rely heavily on `Iframe` and `URL_Length`, whereas UCI models rely on `Sub_Domain` and `Has_External_FormSubmit`.
3. **Zero Monotonic Agreement:** The Spearman rank correlation is $\rho \approx 0.0$, demonstrating that models develop divergent decision boundaries. This explainability finding provides the theoretical explanation for the cross-dataset generalization failure.
